In [1]:
!git clone https://github.com/aliakseizvertouski/bet_analyse

Cloning into 'bet_analyse'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 17 (delta 6), reused 7 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 4.33 MiB | 10.27 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [33]:
bet = pd.read_csv('/content/bet_analyse/bets.csv', sep="|")
bet = bet.copy()

bet["accept_time"] = pd.to_datetime(bet["accept_time"])
bet["create_time"] = pd.to_datetime(bet["create_time"])
bet["settlement_time"] = pd.to_datetime(bet["settlement_time"])
bet.head()

,bet_id,player_id,accept_time,create_time,settlement_time,result,amount,payout,profit,bet_type,...,accepted_bet_odd,item_result,event_stage,accepted_odd,item_amount,item_payout,item_profit,event_id,is_free_bet,settlement_status
0,4580f660-d16a-4ad6-8837-47574564a37f,cf928a91-a17d-42cd-8924-7c3fccd54acb,2022-03-14 00:00:00.165,2022-03-13 23:59:54.503,2022-03-14 00:21:59.745,Return,1.00,1.00,0.00,Ordinar,...,7.50,Return,Live,7.50,1.00,1.00,0.00,c3929913-1eef-4703-ada2-fe08fd47c79d,False,Settled
1,7435c382-e448-4c0b-8b56-a9e547ca6605,cf928a91-a17d-42cd-8924-7c3fccd54acb,2022-03-14 00:00:36.819,2022-03-14 00:00:31.160,2022-03-14 00:10:16.243,Win,0.95,2.48,-1.53,Ordinar,...,2.62,Win,Live,2.62,0.95,2.48,-1.53,c3929913-1eef-4703-ada2-fe08fd47c79d,False,Settled
2,ae36ffa2-0bb5-4bd7-8cf6-e15416ae4fc7,f1866879-4b0f-4a58-aaa9-24e6224cb139,2022-03-14 00:01:07.941,2022-03-14 00:01:07.743,2022-03-14 04:37:24.291,Win,3.51,7.51,-4.00,Ordinar,...,2.14,Win,Prematch,2.14,3.51,7.51,-4.00,7b6aaa15-3b2e-4977-8b1a-59472eeb0817,False,Settled
3,57009dc1-60c5-46f7-acb5-1a45a19983b7,e5efcd9c-aa46-48a1-be0e-9e9f9aa37b53,2022-03-14 00:01:27.927,2022-03-14 00:01:27.927,NaT,NaN,1.00,NaN,NaN,Express,...,0.00,NaN,NaN,0.00,0.00,NaN,NaN,0b912707-3b87-4ec3-886c-2b0f96303977,False,Rejected
4,57009dc1-60c5-46f7-acb5-1a45a19983b7,e5efcd9c-aa46-48a1-be0e-9e9f9aa37b53,2022-03-14 00:01:27.927,2022-03-14 00:01:27.927,NaT,NaN,1.00,NaN,NaN,Express,...,0.00,NaN,NaN,0.00,0.00,NaN,NaN,c3929913-1eef-4703-ada2-fe08fd47c79d,False,Rejected


# Core-level таблицы

### Игроки

In [14]:
players = bet.groupby("player_id").agg(
    bets=("bet_id", "count"),
    turnover=("amount", "sum"),
    profit=("profit", "sum"),
    payout=("payout", "sum"),
    avg_bet=("amount", "mean"),
    avg_odd=("accepted_bet_odd", "mean"),
    std_profit=("profit", "std"),
    free_bets=("is_free_bet", lambda x: (x == 1).sum()),
).reset_index()

players.head()

,player_id,bets,turnover,profit,payout,avg_bet,avg_odd,std_profit,free_bets
0,0038a99b-bdcd-47d7-93ef-812c54804993,28,505.23,-136.86,622.09,18.043929,1.696179,11.432045,0
1,00a8f5ce-3fb0-432b-94df-76ff2693951f,11,26.00,26.00,0.00,2.363636,13.122818,0.504525,0
2,00e94ea9-fa45-4f2f-81db-839caee89f39,108,53.83,17.58,32.57,0.498426,3.425092,0.718604,0
3,01354ecc-625b-477a-8fbf-87ff5e3da1a2,185,614.66,67.94,106.72,3.322486,8.554269,1.059728,0
4,018f4bde-9906-412e-abdb-b9a4fbf320b8,12,395.50,184.50,181.00,32.958333,3.019167,33.846243,0


### Поведение ставок

In [16]:
result_stats = bet.groupby("result").agg(
    count=("bet_id", "count"),
    turnover=("amount", "sum"),
    profit=("profit", "sum")
).reset_index()

result_stats["percent"] = result_stats["count"] / result_stats["count"].sum() * 100

result_stats.head()

,result,count,turnover,profit,percent
0,Cashout,1285,21892.07,1173.97,1.598676
1,Lose,60110,353419.52,353264.52,74.783215
2,Return,713,5783.10,0.00,0.887048
3,TechnicalReturn,11,505.05,0.00,0.013685
4,Win,18260,207606.87,-276849.67,22.717376


### Ивенты

In [21]:
game_stats = bet.groupby("event_id").agg(
    bets=("bet_id", "count"),
    turnover=("amount", "sum"),
    profit=("profit", "sum")
).reset_index()

game_stats.head()


,event_id,bets,turnover,profit
0,0006330b-9987-4284-a7c0-b9a7143fc6c5,5,3.80,1.70
1,000d941c-8cc9-40f8-9af1-0df431783159,24,81.16,-11.06
2,001d12ac-d48e-4b97-a7f7-e99daa224d5f,12,71.15,44.15
3,0024cdb9-25c8-43b7-9748-fdc9c0cec4a4,1,3.00,3.00
4,002acb12-cf57-4768-89b4-aab175820810,22,79.39,62.60


# Расчет показателей

### Основные финансовые показатели за отчетный период

In [25]:
total_turnover = bet["amount"].sum()
total_profit = bet["profit"].sum()
total_payout = bet["payout"].sum()

rtp = total_payout / total_turnover
ggr = -total_profit

kpi_table = pd.DataFrame([{
    "turnover": total_turnover, # ОБОРОТ
    "profit": total_profit,     # ВЫПЛАТЫ ИГРОКАМ
    "payout": total_payout,     # ПРИБЫЛЬ ИГРОКОВ
    "rtp": rtp,                 # Return to Player - ВЫВЕДЕНО ДЕНЕГ ИГРОКАМИ
    "ggr": ggr                  # Gross Gaming Revenue - ПРИБЫЛЬ КАЗИНО
}])

kpi_table

,turnover,profit,payout,rtp,ggr
0,746058.09,77588.82,511432.79,0.685513,-77588.82


### Итоги игровых сессий

In [43]:
wins_looses = bet.groupby("result").size().reset_index(name="count")

wins_looses["percent"] = wins_looses["count"] / wins_looses["count"].sum() * 100

wins_looses

,result,count,percent
0,Cashout,1285,1.598676
1,Lose,60110,74.783215
2,Return,713,0.887048
3,TechnicalReturn,11,0.013685
4,Win,18260,22.717376


### Анализ времени ставок

In [36]:
bet["accept_delay"] = (bet["accept_time"] - bet["create_time"]).dt.total_seconds()         # время между созданием ставки и её подтверждением (cек)
bet["settlement_delay"] = (bet["settlement_time"] - bet["accept_time"]).dt.total_seconds() # время от принятия ставки до её расчёта (cек)

churn_rate = bet[['player_id', "accept_delay", "settlement_delay"]]
churn_rate.head()

,player_id,accept_delay,settlement_delay
0,cf928a91-a17d-42cd-8924-7c3fccd54acb,5.662,1319.580
1,cf928a91-a17d-42cd-8924-7c3fccd54acb,5.659,579.424
2,f1866879-4b0f-4a58-aaa9-24e6224cb139,0.198,16576.350
3,e5efcd9c-aa46-48a1-be0e-9e9f9aa37b53,0.000,NaN
4,e5efcd9c-aa46-48a1-be0e-9e9f9aa37b53,0.000,NaN


In [38]:
accept_delay_stats = pd.DataFrame([{
    "mean_accept_delay": bet["accept_delay"].mean(),
    "min_accept_delay": bet["accept_delay"].min(),
    "max_accept_delay": bet["accept_delay"].max()
}])

accept_delay_stats

,mean_accept_delay,min_accept_delay,max_accept_delay
0,2.754518,0.0,18.942


In [37]:
settlement_delay_stats = pd.DataFrame([{
    "mean_settlement_delay": bet["settlement_delay"].mean(),
    "min_settlement_delay": bet["settlement_delay"].min(),
    "max_settlement_delay": bet["settlement_delay"].max()
}])

settlement_delay_stats

,mean_settlement_delay,min_settlement_delay,max_settlement_delay
0,12299.779961,5.434,1648497.98


### Проверка задержек обработки ставок

In [51]:
delay_check = bet[
    (bet["accept_delay"] > 10) &
    (bet["result"] == "TechnicalReturn")
][["bet_id", "accept_delay", "result"]]

delay_check.head()  # задержки есть, но игроки заканчивают игровую сессию - все ок

,bet_id,accept_delay,result


#Customer Churn Analysis

In [53]:
free_bet_players = (
    bet[['player_id', 'bet_id', 'is_free_bet']]
    .drop_duplicates()
    .groupby(['player_id', 'is_free_bet'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "non_free", True: "free"})
)

free_bet_players = free_bet_players[
    (free_bet_players["non_free"] == 0) &
    (free_bet_players["free"] > 10)
]

free_bet_players

is_free_bet,non_free,free
player_id,,
c02e1329-57f6-42af-a478-216c825a9ddc,0,31
f986aad8-d457-482e-aaee-675a37b70b78,0,14


In [55]:
freeloaders = free_bet_players.reset_index()
freeloaders = freeloaders['player_id'].nunique() / bet['player_id'].nunique() * 100

print (f'итого халявщиков: {freeloaders} %')

итого халявщиков: 0.2544529262086514 %


перепадает им что-нибудь?

In [ ]:
profit_by_player = bet.groupby("player_id")["profit"].sum().reset_index()

freeloaders_profit = free_bet_players.merge(profit_by_player, on="player_id", how="left")
freeloaders_profit

неа( не стоит надеяться на фриспины